### The purpose of this notebook is to recreate the baseline visual instruction tuning pipeline in the LLaVA paper

## This is how our baseline workload looks like



## Project Structure

### Please create such directories

In [ ]:
import os
from pathlib import Path

# Get the directory where the script is located
#TODO:
SCRIPT_DIR = ""

# The project root is one level up from 'scripts/'
PROJECT_ROOT = Path(SCRIPT_DIR).parent

# Define your data paths relative to the project root
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "clip_embeddings"

# Ensure directories exist
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")

Project Root: /home/lhf_hongfu_gmail_com/tunix/CS159FinalProject


In [2]:
#UTILITY FUNCTIONS
def show_hbm_usage():
  """Displays memory usage per device."""
  fmt_size = functools.partial(humanize.naturalsize, binary=True)

  for d in jax.local_devices():
    stats = d.memory_stats()
    used = stats["bytes_in_use"]
    limit = stats["bytes_limit"]
    print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")

## Our Datasets 

### For baseline, we are using COCO for both stage 1 and stage 2, instead of CC3M and LLaVA-instruct-158k, for simplicity and debugging

In [ ]:
# Download the COCO 2017 dataset and annotations 

! mkdir -p data/raw
! cd data/raw

! wget http://images.cocodataset.org/zips/train2017.zip
! wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

! unzip train2017.zip
! unzip annotations_trainval2017.zip

In [ ]:
# Match the COCO captions with the corresponding images and save in a new JSON file for stage 1 training.
!python prepare_stage1_dataset.py

In [ ]:
# We are adding a simple instruction to the captions to make them more suitable for training the projector in stage 1.
# Make sure to execute this at project root directory (CS159FinalProject/) since the paths in the script are relative to it.
!python convert_alignment_format.py

In [ ]:
# Loggin to HF

import os

import json
import jax
import jax.numpy as jnp
import numpy as np

from PIL import Image
import os
import kagglehub

try:
  from google.colab import userdata
  USE_COLAB = True

  %pip uninstall -q wandb -y  # wandb is glitchy with tunix in colab

#TODO: Optional HF token and kaggle credentials for colab. If not set, will skip login and rely on kagglehub to download datasets.
  os.environ["HF_TOKEN"] = 
  os.environ["KAGGLE_USERNAME"] = 
  os.environ["KAGGLE_KEY"] = 
except:
  USE_COLAB = False

  from dotenv import load_dotenv
  load_dotenv()
  print("Using env vars to login")

  import nest_asyncio
  nest_asyncio.apply()
  print("nest_asyncio applied")

  # Only using wandb on TPU VM because it has strange bugs on Colab
  !pip install -q wandb
  import wandb
  # Check if WANDB_API_KEY is set before logging in
  if "WANDB_API_KEY" in os.environ and os.environ["WANDB_API_KEY"]:
      wandb.login(key=os.environ["WANDB_API_KEY"])
  else:
      print("WANDB_API_KEY not found. Skipping wandb login.")

if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    hf_token = os.environ["HF_TOKEN"]
    !hf auth login --token "$hf_token"
else:
    print("HF_TOKEN not found. Skipping Hugging Face login.")

Using env vars to login
nest_asyncio applied
WANDB_API_KEY not found. Skipping wandb login.


HF_TOKEN not found. Skipping Hugging Face login.


In [2]:
# ====== Sharding ======
# Adjust mesh based on your TPU memory and model size.
NUM_TPUS = len(jax.devices())
if NUM_TPUS == 8:
  MESH_COUNTS = (2, 4)
elif NUM_TPUS == 4:
  MESH_COUNTS = (1, 4)
elif NUM_TPUS == 1:
  MESH_COUNTS = (1, 1)
else:
  raise ValueError(f"Unsupported number of TPUs: {NUM_TPUS}")

MESH = [
    MESH_COUNTS,
    ("fsdp", "tp"),
]

# Get HF access to llama 3.2 1B Instruct model files

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [30]:
# We need to bypass the standard transformer loader as it would have a stuck progress bar on TPU. Setting this environment variable disables the progress bar and allows the model to load successfully.
# First let's download Llama
import model as llama_model
import params as llama_params



In [ ]:
# Export HF API Key to get access to the model files. Make sure to set HF_TOKEN in your environment variables before running this cell.

#TODO: Again, set HF API KEY
os.environ["HF_TOKEN"] = 

In [32]:
! HF_HUB_ENABLE_HF_TRANSFER

!hf download meta-llama/Llama-3.2-1B-Instruct \
  --include "config.json" "generation_config.json" \
            "tokenizer.json" "tokenizer_config.json" \
            "special_tokens_map.json" \
            "model.safetensors" "model.safetensors.index.json" \
  --local-dir CS159FinalProject/temp/hf_models/Llama-3.2-1B-Instruct

/bin/bash: line 1: HF_HUB_ENABLE_HF_TRANSFER: command not found


/home/lhf_hongfu_gmail_com/miniconda3/envs/tpujax/lib/python3.13/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Fetching 6 files: 100%|█████████████████████████| 6/6 [00:00<00:00, 5576.30it/s]
/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/scripts/CS159FinalProject/temp/hf_models/Llama-3.2-1B-Instruct


In [33]:
# Choose the correct config for the model you downloaded. This should match the config.json file in the model directory.
# Config is used to define the architecture of the model and must match the parameters used during training. 
# If you downloaded a different model, make sure to update this config accordingly.
cfg = llama_model.ModelConfig.llama3p2_1b_instruct()

In [34]:
llama_dir = "CS159FinalProject/temp/hf_models/Llama-3.2-1B-Instruct"
mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0]))
with mesh:
    llama3_1b = llama_params.create_model_from_safe_tensors(
        file_dir=llama_dir,
        config=cfg,
        mesh=mesh,
        dtype=jnp.bfloat16,
    )

In [35]:
# WE can display the model architecture using nnx.display. This will show the layers and parameters of the model, which can be useful for debugging and understanding the model structure.
from flax import nnx
nnx.display(llama3_1b)

In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    llama_dir,
    local_files_only=True,
)

download CLIP locally
→ load FlaxCLIPModel from local files
→ keep only vision tower for feature extraction
→ return processor + model + vision hidden size

In [37]:
# Now the same deal for CLIP. We need the vision encoder to extract image features for the projector training in stage 1, and also for the multimodal finetuning in stage 2.
from clip_helpers import download_clip_flax

clip_dir = download_clip_flax()
print(clip_dir)

/home/lhf_hongfu_gmail_com/hf_models/clip-vit-base-patch32
/home/lhf_hongfu_gmail_com/hf_models/clip-vit-base-patch32


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 11016.93it/s]


In [38]:
from clip_helpers import build_clip_vision_tower

clip_bundle = build_clip_vision_tower()
print("CLIP hidden size:", clip_bundle.hidden_size)
print("Image size:", clip_bundle.image_size)
print("Patch size:", clip_bundle.patch_size)

Some weights of the model checkpoint at /home/lhf_hongfu_gmail_com/hf_models/clip-vit-base-patch32 were not used when initializing FlaxCLIPVisionModel: {('text_model', 'encoder', 'layers', '3', 'mlp', 'fc2', 'kernel'), ('text_model', 'encoder', 'layers', '6', 'layer_norm2', 'bias'), ('text_model', 'encoder', 'layers', '4', 'layer_norm2', 'bias'), ('text_model', 'encoder', 'layers', '3', 'self_attn', 'k_proj', 'kernel'), ('text_model', 'encoder', 'layers', '3', 'mlp', 'fc1', 'kernel'), ('text_model', 'encoder', 'layers', '7', 'self_attn', 'out_proj', 'bias'), ('text_model', 'encoder', 'layers', '6', 'self_attn', 'q_proj', 'kernel'), ('text_model', 'encoder', 'layers', '5', 'self_attn', 'q_proj', 'kernel'), ('text_model', 'encoder', 'layers', '0', 'layer_norm2', 'scale'), ('text_model', 'encoder', 'layers', '10', 'layer_norm1', 'bias'), ('text_model', 'encoder', 'layers', '7', 'mlp', 'fc1', 'bias'), ('text_model', 'encoder', 'layers', '3', 'self_attn', 'v_proj', 'kernel'), ('text_model',

CLIP hidden size: 768
Image size: 224
Patch size: 32


In [39]:
#Sanity Check
from clip_helpers import load_clip_flax_local
clip_bundle = load_clip_flax_local()
print(type(clip_bundle.model))

Some weights of the model checkpoint at /home/lhf_hongfu_gmail_com/hf_models/clip-vit-base-patch32 were not used when initializing FlaxCLIPVisionModel: {('text_model', 'encoder', 'layers', '3', 'mlp', 'fc2', 'kernel'), ('text_model', 'encoder', 'layers', '6', 'layer_norm2', 'bias'), ('text_model', 'encoder', 'layers', '4', 'layer_norm2', 'bias'), ('text_model', 'encoder', 'layers', '3', 'self_attn', 'k_proj', 'kernel'), ('text_model', 'encoder', 'layers', '3', 'mlp', 'fc1', 'kernel'), ('text_model', 'encoder', 'layers', '7', 'self_attn', 'out_proj', 'bias'), ('text_model', 'encoder', 'layers', '6', 'self_attn', 'q_proj', 'kernel'), ('text_model', 'encoder', 'layers', '5', 'self_attn', 'q_proj', 'kernel'), ('text_model', 'encoder', 'layers', '0', 'layer_norm2', 'scale'), ('text_model', 'encoder', 'layers', '10', 'layer_norm1', 'bias'), ('text_model', 'encoder', 'layers', '7', 'mlp', 'fc1', 'bias'), ('text_model', 'encoder', 'layers', '3', 'self_attn', 'v_proj', 'kernel'), ('text_model',

<class 'transformers.models.clip.modeling_flax_clip.FlaxCLIPVisionModel'>


In [40]:
model = clip_bundle.model

### Sanity Check
from PIL import Image
import requests
from transformers import AutoProcessor, FlaxCLIPModel

model = FlaxCLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="np")

image_features = model.get_image_features(**inputs)

In [41]:
# Check available JAX devices, essential for setting up mesh geometry
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0)]

In [42]:
# Stage 1: Feature Alignment


In [43]:
from flax import nnx
import jax
import jax.numpy as jnp


class VisionProjector(nnx.Module):
    def __init__(self, in_dim: int=768, out_dim: int = 2048, *, rngs: nnx.Rngs):
        self.proj = nnx.Linear(
            in_features=in_dim,
            out_features=out_dim,
            use_bias=False,
            rngs=rngs,
        )

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # x: [B, N_vis, D_clip]
        return self.proj(x)  # [B, N_vis, D_llama]
    
    
def serialize_stage1_sample(tokenizer, sample, max_len=128):
    instruction = sample["instruction"]
    response = sample["response"]

    prefix = f"USER: {instruction}\nASSISTANT:"
    full_text = prefix + " " + response

    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]

    full_ids = full_ids[:max_len]
    labels = full_ids.copy()

    prefix_len = min(len(prefix_ids), len(labels))
    labels[:prefix_len] = [-100] * prefix_len

    return {
        "image": sample["image"],
        "input_ids": full_ids,
        "labels": labels,
    }

In [44]:
def masked_cross_entropy_loss(logits, labels):
    # logits: [B, L, V]
    # labels: [B, L]
    #B: Batch size, L: Sequence length, V: Vocabulary size
    shift_logits = logits[:, :-1, :] #Select the logits for all tokens except the last one, since we are predicting the next token at each position.
    shift_labels = labels[:, 1:] #Shift the labels to the left by one position, so that each position's label corresponds to the next token that the model should predict.

    valid = shift_labels != -100
    safe_labels = jnp.where(valid, shift_labels, 0)

    log_probs = jax.nn.log_softmax(shift_logits, axis=-1)
    token_logp = jnp.take_along_axis(
        log_probs,
        safe_labels[..., None],
        axis=-1
    ).squeeze(-1)

    loss = -jnp.sum(token_logp * valid) / jnp.maximum(jnp.sum(valid), 1)
    return loss

In [45]:
# During training, confirm these dimensions: 

#vision_feats: [B, N_vis, D_clip]
#vis_embeds: [B, N_vis, 2048]
#text_embeds: [B, T, 2048]
#input_embeds: [B, N_vis + T, 2048]
#logits: [B, N_vis + T, vocab_size]

## Stage 1 Orchestration

Now let’s wire the whole thing.

The best first orchestration is:

1. load processed Stage 1 JSON
2. tokenize text examples
3. precompute CLIP vision features
4. freeze CLIP + Llama
5. train projector only

In [46]:
#Compute stage 1 tokenized dataset

import json
import os

def build_tokenized_stage1_dataset(tokenizer, input_json, output_json, max_len=128):
    with open(input_json, "r") as f:
        data = json.load(f)

    processed = []
    for sample in data:
        ex = serialize_stage1_sample(tokenizer, sample, max_len=max_len)
        processed.append(ex)

    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(processed, f)

In [ ]:
#Build tokenized dataset for stage 1 training. This will create a new JSON file where each sample contains the tokenized input IDs and labels, ready for training the vision projector. Make sure to run this after downloading the COCO dataset and preparing the alignment JSON.

#TODO: Modify this accordingly
# 1. Define Paths (Ensuring consistency with your directory structure)
ALIGNMENT_JSON = "/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/stage1_alignment/alignment_chat.json"
TOKENIZED_JSON = "/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/stage1_alignment/alignment_tokenized.json"
IMAGE_ROOT = "/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/raw/train2017" # Adjust if your images are in 'images/'
FEATURE_DIR = "/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/clip_embeddings"
MANIFEST_JSON = "/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/stage1_alignment/stage1_manifest.json"

''' #Uncomment this block if you need to re-run the tokenization step. Make sure to run this after preparing the alignment JSON.
# --- STEP 1: Tokenize the text ---
print("Step 1: Tokenizing dataset...")
build_tokenized_stage1_dataset(
    tokenizer=tokenizer,
    input_json=ALIGNMENT_JSON,
    output_json=TOKENIZED_JSON,
    max_len=128
)
'''

' #Uncomment this block if you need to re-run the tokenization step. Make sure to run this after preparing the alignment JSON.\n# --- STEP 1: Tokenize the text ---\nprint("Step 1: Tokenizing dataset...")\nbuild_tokenized_stage1_dataset(\n    tokenizer=tokenizer,\n    input_json=ALIGNMENT_JSON,\n    output_json=TOKENIZED_JSON,\n    max_len=128\n)\n'

In [48]:
def make_clip_feature_fn(clip_bundle):
    
    '''Returns a function that takes in raw pixel values and returns the CLIP vision features. 
    This function is JIT-compiled for efficiency.'''
    
    params = clip_bundle.model.params

    @jax.jit
    def get_features(pixel_values):
        outputs = clip_bundle.model(
            pixel_values=pixel_values,
            params=params,
            output_hidden_states=True,
        )
        return outputs.hidden_states[-2]

    return get_features


In [ ]:
import json
import os
import numpy as np
import jax.numpy as jnp
from PIL import Image


def precompute_clip_features_jitted(
    clip_bundle,
    tokenized_json,
    image_root,
    output_dir,
):
    with open(tokenized_json, "r") as f:
        data = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    get_features_compiled = make_clip_feature_fn(clip_bundle) #Get the JIT-compiled function for feature extraction

    print(f"Extracting features for {len(data)} images...")

    for i, sample in enumerate(data):
        try:
            image_path = os.path.join(image_root, sample["image"])
            img = Image.open(image_path).convert("RGB")

            clip_inputs = clip_bundle.processor(
                images=img,
                return_tensors="np",
            )

            pixel_values = jnp.array(clip_inputs["pixel_values"])

            # [B, seq_len, hidden_dim]
            hidden_states_penultimate = get_features_compiled(pixel_values) #Extract features using the JIT-compiled function

            # save single example: [seq_len, hidden_dim]
            vision_feats = np.array(hidden_states_penultimate[0])

            save_path = os.path.join(
                output_dir,
                sample["image"].replace(".jpg", ".npy"),
            )
            np.save(save_path, vision_feats)

            if i % 100 == 0:
                print(f"Processed {i}/{len(data)}...")

        except Exception as e:
            print(f"Error on {sample['image']}: {e}")
            
# Precompute CLIP features for all images in the tokenized dataset. This will save the features as .npy files in the specified output directory. Make sure to run this after tokenizing the dataset and before training the projector, as the training loop will load these precomputed features.
precompute_clip_features_jitted(
    clip_bundle=clip_bundle,
    tokenized_json=TOKENIZED_JSON,
    image_root=IMAGE_ROOT,
    output_dir=FEATURE_DIR,
)   

Extracting features for 591753 images...
Processed 0/591753...
Processed 100/591753...
Processed 200/591753...
Processed 300/591753...
Processed 400/591753...
Processed 500/591753...
Processed 600/591753...
Processed 700/591753...
Processed 800/591753...
Processed 900/591753...
Processed 1000/591753...
Processed 1100/591753...
Processed 1200/591753...
Processed 1300/591753...
Processed 1400/591753...
Processed 1500/591753...
Processed 1600/591753...
Processed 1700/591753...
Processed 1800/591753...
Processed 1900/591753...
Processed 2000/591753...
Processed 2100/591753...
Processed 2200/591753...
Processed 2300/591753...
Processed 2400/591753...
Processed 2500/591753...
Processed 2600/591753...
Processed 2700/591753...
Processed 2800/591753...
Processed 2900/591753...
Processed 3000/591753...
Processed 3100/591753...
Processed 3200/591753...
Processed 3300/591753...
Processed 3400/591753...
Processed 3500/591753...
Processed 3600/591753...
Processed 3700/591753...
Processed 3800/591753

In [ ]:
#Double check that we are using FlaxCLIPVISIONModel
print(type(clip_bundle.model))
print(clip_bundle.model.__class__.__name__)
print(type(clip_bundle.model.config))
print(clip_bundle.model.config.model_type)

<class 'transformers.models.clip.modeling_flax_clip.FlaxCLIPVisionModel'>
FlaxCLIPVisionModel
<class 'transformers.models.clip.configuration_clip.CLIPVisionConfig'>
clip_vision_model


In [ ]:
# 3. Build trainable examples
#Build the manifest for stage 1 training. This manifest will be used by the training loop to load the precomputed CLIP features and the tokenized text inputs for each training example.

from build_stage1_manifest import build_stage1_manifest
# --- STEP 3: Build Manifest ---
print("\nStep 3: Building final training manifest...")
build_stage1_manifest(
    tokenized_json=TOKENIZED_JSON,
    feature_dir=FEATURE_DIR,
    output_json=MANIFEST_JSON
)

print(f"\nSuccess! Manifest created at: {MANIFEST_JSON}")


Step 3: Building final training manifest...

Success! Manifest created at: /home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/stage1_alignment/stage1_manifest.json


In [ ]:
# Check manifest is valid
with open(MANIFEST_JSON, "r") as f:
    sample_manifest = json.load(f)[0]
    
print("Manifest Sample Check:")
print(f" - Vision Path: {sample_manifest['vision_path']}")
print(f" - Vision Exists: {os.path.exists(sample_manifest['vision_path'])}")
print(f" - Token Count: {len(sample_manifest['input_ids'])}")

Manifest Sample Check:
 - Vision Path: /home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/clip_embeddings/000000203564.npy
 - Vision Exists: True
 - Token Count: 25


In [ ]:
# 4 Collator

import numpy as np


def pad_list(x, length, pad_value):
    return x + [pad_value] * (length - len(x))


def stage1_collate_fn(batch):
    max_text_len = max(len(x["input_ids"]) for x in batch)

    vision_feats = []
    input_ids = []
    labels = []

    for sample in batch:
        vf = np.load(sample["vision_path"])   # [N_vis, D_clip] = [50, 768]
        vision_feats.append(vf)

        input_ids.append(pad_list(sample["input_ids"], max_text_len, 0))
        labels.append(pad_list(sample["labels"], max_text_len, -100))

    vision_feats = np.stack(vision_feats, axis=0)    # assumes same CLIP seq len
    input_ids = np.array(input_ids, dtype=np.int32)
    labels = np.array(labels, dtype=np.int32)

    return {
        "vision_feats": vision_feats, #Dimension: [B, N_vis, D_clip] = [Batch size, Number of visual tokens, CLIP feature dimension] [B,50,768]
        "input_ids": input_ids, #Dimension: [B, T] = [Batch size, Text sequence length] [B,128]
        "labels": labels,
    }

In [ ]:
#5 Build multimodal batch inside training loop, since we need to feed the raw images to the CLIP vision tower and get the projected embeddings on the fly for each batch. 

import jax.numpy as jnp


def make_multimodal_inputs(llama_model, projector, batch):
    vision_feats = jnp.array(batch["vision_feats"])
    input_ids = jnp.array(batch["input_ids"])
    labels = jnp.array(batch["labels"])

    vis_embeds = projector(vision_feats)                # [B, N_vis, D]
    txt_embeds = llama_model.embedder.encode(input_ids) # [B, T, D]

    input_embeds = jnp.concatenate([vis_embeds, txt_embeds], axis=1)

    bsz, n_vis, _ = vis_embeds.shape
    _, tlen = input_ids.shape
    seq_len = n_vis + tlen

    vis_labels = -100 * jnp.ones((bsz, n_vis), dtype=jnp.int32)
    full_labels = jnp.concatenate([vis_labels, labels], axis=1)

    positions = jnp.broadcast_to(
        jnp.arange(seq_len)[None, :],
        (bsz, seq_len)
    )

    attention_mask = jnp.tril(
        jnp.ones((seq_len, seq_len), dtype=bool)
    )
    attention_mask = jnp.broadcast_to(
        attention_mask[None, :, :],
        (bsz, seq_len, seq_len)
    )

    return {
        "input_embeds": input_embeds,
        "labels": full_labels,
        "positions": positions,
        "attention_mask": attention_mask,
    }

In [ ]:
# 6 Optimizer and training step
import optax
from flax import nnx
import jax
#Freeze the CLIP vision tower and LLaMA model during stage 1 training, only train the projector.

tx = optax.adamw(learning_rate=1e-3, weight_decay=0.0)

projector = VisionProjector(
    in_dim=clip_bundle.hidden_size,
    out_dim=llama3_1b.config.embed_dim,
    rngs=nnx.Rngs(0),
)

projector_graphdef, projector_state = nnx.split(projector) #Split the projector module into a graph definition (which contains the structure of the module) and a state (which contains the parameters). This allows us to merge the state back into the module during the training step, which is necessary for JIT compilation.
opt_state = tx.init(projector_state)

In [ ]:
@nnx.jit # Use nnx.jit for compatibility with nnx modules
def train_step(projector_state, opt_state, batch, projector_graphdef, llama_state, llama_graphdef):
    def loss_fn(proj_state):
        # Merge states back into modules inside the loss function
        projector = nnx.merge(projector_graphdef, proj_state)
        llama_model = nnx.merge(llama_graphdef, llama_state)

        mm = make_multimodal_inputs(llama_model, projector, batch) #Build the multimodal inputs for the current batch by passing the raw vision features through the projector and encoding the text inputs using the LLaMA embedder. This will give us the input embeddings, labels, positions, and attention mask that we need to feed into the LLaMA model for the forward pass.

        # Call the Llama forward pass
        logits, _ = llama_model.forward_from_embeddings(
            input_embeds=mm["input_embeds"],
            positions=mm["positions"],
            cache=None,
            attention_mask=mm["attention_mask"],
        )

        return masked_cross_entropy_loss(logits, mm["labels"])

    loss, grads = jax.value_and_grad(loss_fn)(projector_state)
    updates, opt_state = tx.update(grads, opt_state, projector_state)
    projector_state = optax.apply_updates(projector_state, updates)
    return projector_state, opt_state, loss

In [ ]:
import random
import json


def iterate_minibatches(data, batch_size):
    idxs = list(range(len(data)))
    random.shuffle(idxs)

    for i in range(0, len(idxs), batch_size):
        batch = [data[j] for j in idxs[i:i + batch_size]]
        yield stage1_collate_fn(batch)


def run_stage1_training(
    manifest_json,
    llama_model,
    clip_bundle,
    num_epochs=1,
    batch_size=2,
):
    global projector_state, opt_state #Access the global projector state and optimizer state variables that we initialized earlier. This allows us to update the projector parameters across training steps.

    with open(manifest_json, "r") as f:
        manifest = json.load(f)

    step = 0
    for epoch in range(num_epochs):
        for batch in iterate_minibatches(manifest, batch_size=batch_size):
            projector_state, opt_state, loss = train_step(
                projector_state,
                opt_state,
                batch,
                projector_graphdef,
                llama_model,
            )

            if step % 10 == 0:
                print(f"epoch={epoch} step={step} loss={float(loss):.4f}")
            step += 1

In [ ]:
run_stage1_training(
    manifest_json=MANIFEST_JSON,
    llama_model=llama3_1b,
    clip_bundle=clip_bundle,
    num_epochs=1,
    batch_size=4,
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/lhf_hongfu_gmail_com/tunix/CS159FinalProject/data/processed/clip_embeddings/000000352498.npy'

## Step 2: Multimodal Finetuning


1. initialize projector from Stage 1
2. keep CLIP frozen
3. keep projector trainable
4. unfreeze Llama
5. train on image + instruction + response instead of simple caption prompts

In [ ]:
#2. Serilize multimodal instruction data 

import pickle
projector = VisionProjector(
    in_dim=clip_bundle.hidden_size,
    out_dim=llama_model.config.embed_dim,
    rngs=nnx.Rngs(0),
)
#So instead of keeping only projector_state trainable, you now split both models.
llama_graphdef, llama_state = nnx.split(llama_model)
projector_graphdef, projector_state = nnx.split(projector)

# restore saved projector_state from Stage 1
with open(PROJECTOR_STATE_PATH, "rb") as f:
    projector_state = pickle.load(f)
    
def serialize_stage2_sample(tokenizer, sample, max_len=256):
    instruction = sample["instruction"]
    response = sample["response"]

    prefix = f"USER: {instruction}\nASSISTANT:"
    full_text = prefix + " " + response

    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]

    full_ids = full_ids[:max_len]
    labels = full_ids.copy()

    prefix_len = min(len(prefix_ids), len(labels))
    labels[:prefix_len] = [-100] * prefix_len

    return {
        "image": sample["image"],
        "input_ids": full_ids,
        "labels": labels,
    }

In [ ]:
#3: Multimodal input builder

def make_multimodal_inputs(llama_model, projector, batch):
    vision_feats = jnp.array(batch["vision_feats"])
    input_ids = jnp.array(batch["input_ids"])
    labels = jnp.array(batch["labels"])

    vis_embeds = projector(vision_feats)
    txt_embeds = llama_model.embedder.encode(input_ids)

    input_embeds = jnp.concatenate([vis_embeds, txt_embeds], axis=1)

    bsz, n_vis, _ = vis_embeds.shape
    _, tlen = input_ids.shape
    seq_len = n_vis + tlen

    vis_labels = -100 * jnp.ones((bsz, n_vis), dtype=jnp.int32)
    full_labels = jnp.concatenate([vis_labels, labels], axis=1)

    positions = jnp.broadcast_to(
        jnp.arange(seq_len)[None, :],
        (bsz, seq_len)
    )

    attention_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))
    attention_mask = jnp.broadcast_to(attention_mask[None, :, :], (bsz, seq_len, seq_len))

    return {
        "input_embeds": input_embeds,
        "labels": full_labels,
        "positions": positions,
        "attention_mask": attention_mask,
    }
    

In [ ]:
tx = optax.adamw(
    learning_rate=2e-5,
    weight_decay=0.0,
)

opt_state = tx.init({
    "projector": projector_state,
    "llama": llama_state,
})

In [ ]:
@jax.jit
def train_step_stage2(
    projector_state,
    llama_state,
    opt_state,
    batch,
    projector_graphdef,
    llama_graphdef,
):
    def loss_fn(projector_state, llama_state):
        projector = nnx.merge(projector_graphdef, projector_state)
        llama_model = nnx.merge(llama_graphdef, llama_state)

        mm = make_multimodal_inputs(llama_model, projector, batch)

        logits, _ = llama_model.forward_from_embeddings(
            input_embeds=mm["input_embeds"],
            positions=mm["positions"],
            cache=None,
            attention_mask=mm["attention_mask"],
        )

        return masked_cross_entropy_loss(logits, mm["labels"])

    loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1))(projector_state, llama_state)
    projector_grads, llama_grads = grads

    updates, opt_state = tx.update(
        {"projector": projector_grads, "llama": llama_grads},
        opt_state,
        {"projector": projector_state, "llama": llama_state},
    )

    new_projector_state = optax.apply_updates(projector_state, updates["projector"])
    new_llama_state = optax.apply_updates(llama_state, updates["llama"])

    return new_projector_state, new_llama_state, opt_state, loss